# Credit Card Default Risk Scorecard

Portfolio analytics notebook using UCI's historical Default of Credit Card Clients data. Demographic fields are reviewed descriptively and excluded from model inputs. The results are an educational example, not current Australian credit policy.


## 1. Reproducible setup


In [ ]:
from pathlib import Path
import sqlite3
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "credit_risk.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "credit_risk.py").exists():
    raise FileNotFoundError("Run this notebook from the project checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from credit_risk import (
    DEMOGRAPHIC_COLUMNS, PAYMENT_STATUS_COLUMNS, TARGET, WoEEncoder,
    approval_policy_table, build_scorecard, ks_statistic, load_uci_dataset,
    score_from_probability,
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")
print(f"Project root: {PROJECT_ROOT}")


## 2. Load, clean and explore the UCI data


In [ ]:
data = load_uci_dataset(PROJECT_ROOT / "data" / "raw")
assert len(data) == 30_000
assert data[TARGET].isin([0, 1]).all()
assert not data.isna().any().any()
print(f"Rows: {len(data):,} | fields: {data.shape[1]} | default rate: {data[TARGET].mean():.2%}")
display(data.head())
display(data.dtypes.astype(str).to_frame("dtype"))

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Paid", "Default"], data[TARGET].value_counts(normalize=True).sort_index(),
       color=["#4C78A8", "#E45756"])
ax.set(title="Next-month outcome mix", ylabel="Share of accounts", ylim=(0, 1))
plt.tight_layout()
plt.show()


## 3. Portfolio summaries with SQLite


In [ ]:
sql_text = (PROJECT_ROOT / "sql" / "eda_queries.sql").read_text(encoding="utf-8")
queries = [query.strip() for query in sql_text.split(";") if query.strip()]
assert len(queries) == 3
with sqlite3.connect(":memory:") as connection:
    data.to_sql("credit_clients", connection, index=False, if_exists="replace")
    for number, query in enumerate(queries, start=1):
        print(f"SQLite summary {number}")
        display(pd.read_sql_query(query, connection))


## 4. Stratified split and training-only WoE/IV screening


In [ ]:
excluded = {TARGET, *DEMOGRAPHIC_COLUMNS}
model_features = [column for column in data.columns if column not in excluded]
X, y = data[model_features].copy(), data[TARGET].copy()
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_valid, y_train_valid, test_size=0.25,
    random_state=RANDOM_STATE, stratify=y_train_valid
)
categorical_features = [column for column in PAYMENT_STATUS_COLUMNS if column in X_train]
encoder = WoEEncoder(
    categorical_columns=categorical_features, max_bins=5,
    min_category_count=30, smoothing=0.5
).fit(X_train, y_train)
iv_results = encoder.iv_table_.copy()
iv_results["selected"] = iv_results["iv"] > 0.02
selected_features = iv_results.loc[iv_results["selected"], "feature"].tolist()
if not selected_features:
    raise ValueError("No features passed the training-only IV screen.")
print(f"Selected {len(selected_features)} of {len(model_features)} behaviour features (IV > 0.02).")
display(iv_results)
X_train_woe = encoder.transform(X_train)[selected_features]
X_validation_woe = encoder.transform(X_validation)[selected_features]


## 5. Logistic scorecard and XGBoost benchmark


In [ ]:
scorecard_model = LogisticRegression(
    C=1.0, max_iter=2_000, solver="lbfgs", random_state=RANDOM_STATE
).fit(X_train_woe, y_train)

numeric_features = [column for column in model_features if column not in categorical_features]
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("payment_status", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ], remainder="drop"
)
xgb_pipeline = Pipeline([
    ("preprocess", xgb_preprocessor),
    ("model", XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0,
        eval_metric="logloss", tree_method="hist", n_jobs=4,
        random_state=RANDOM_STATE
    )),
]).fit(X_train, y_train)
validation_probabilities = {
    "Logistic scorecard": scorecard_model.predict_proba(X_validation_woe)[:, 1],
    "XGBoost": xgb_pipeline.predict_proba(X_validation)[:, 1],
}
validation_metrics = pd.DataFrame([
    {"model": name, "validation ROC AUC": roc_auc_score(y_validation, prob),
     "validation KS": ks_statistic(y_validation, prob)}
    for name, prob in validation_probabilities.items()
])
display(validation_metrics)


## 6. Approval-policy trade-offs and final test evaluation


In [ ]:
validation_pd = validation_probabilities["Logistic scorecard"]
threshold_grid = np.linspace(0.005, min(0.60, float(validation_pd.max())), 120)
validation_policy = approval_policy_table(y_validation, validation_pd, threshold_grid.tolist())
feasible = validation_policy.loc[
    (validation_policy["bad rate among approved"] <= 0.10)
    & (validation_policy["approval rate"] > 0)
]
if feasible.empty:
    chosen_policy = validation_policy.loc[validation_policy["approval rate"] > 0].sort_values(
        "bad rate among approved"
    ).iloc[0]
    validation_target_met = False
else:
    chosen_policy = feasible.sort_values("PD approval threshold").iloc[-1]
    validation_target_met = True
chosen_threshold = float(chosen_policy["PD approval threshold"])
print(f"Validation-selected PD threshold: {chosen_threshold:.3f}; 10% bad-rate target met: {validation_target_met}")

test_probabilities = {
    "Logistic scorecard": scorecard_model.predict_proba(encoder.transform(X_test)[selected_features])[:, 1],
    "XGBoost": xgb_pipeline.predict_proba(X_test)[:, 1],
}
test_metrics = pd.DataFrame([
    {"model": name, "test ROC AUC": roc_auc_score(y_test, prob),
     "test KS": ks_statistic(y_test, prob)}
    for name, prob in test_probabilities.items()
])
display(test_metrics)
for name, probability in test_probabilities.items():
    matrix = confusion_matrix(y_test, probability >= 0.50, labels=[0, 1])
    print(f"{name} confusion matrix at PD 0.50")
    display(pd.DataFrame(matrix, index=["Actual paid", "Actual default"],
                         columns=["Predicted paid", "Predicted default"]))

policy_test = approval_policy_table(
    y_test, test_probabilities["Logistic scorecard"],
    [0.05, 0.10, chosen_threshold, 0.20, 0.30, 0.40, 0.50]
)
display(policy_test.style.format({
    "PD approval threshold": "{:.3f}", "approval rate": "{:.1%}",
    "bad rate among approved": "{:.1%}",
    "share of all defaults approved": "{:.1%}"
}))


## 7. Additive points scorecard


In [ ]:
scorecard = build_scorecard(
    scorecard_model, encoder, selected_features,
    base_score=600, base_good_odds=50, points_to_double_odds=20
)
test_score = score_from_probability(test_probabilities["Logistic scorecard"])
print(f"Illustrative test score range: {test_score.min():.0f} to {test_score.max():.0f}")
display(scorecard.head(30).style.format({
    "woe": "{:.4f}", "coefficient": "{:.4f}", "points": "{:.2f}"
}, na_rep=""))


## Interpretation and limitations

A stricter PD cut-off generally reduces approvals and observed bad rate while leaving more defaults outside the approved group. The selected threshold is an illustrative validation-set sensitivity rule. This 2005 Taiwan existing-cardholder sample and random split do not establish current Australian applicant performance, calibration, fairness or suitability for lending decisions.
